# Introducing interfacial thermal resistance $R_i$ in micro-layer simulations

Interfacial thermal resistances $R_i$ can be included in **two distinct regions** within the micro-layer (ML) simulation framework.

> **Note:** You can independently activate one or both regions within the same dataset.

---

## Meso region — `adjust_meso_ML` (in the **TCL** module)

This option adds $R_i$ **only to cells identified as part of the meso region (`Nmeso`)**, not necessarily affecting the entire ML domain (depending on Nmeso and thickness of ML).  By consequence, this option may induce a **heat flux discontinuity** (flux jump) at the meso–macro transition.

**Parameters:**
- `adjust_meso_ML` — flag to activate the meso-level thermal resistance  
- `Ri_thermal 3.7e-6` — thermal resistance value  
- `thetaC_tcl $theta` — local slope threshold beyond which the thermal resistance is applied  

---

## Macro region — `Ri_liq` (in the **Echange_contact_VDF_FT_Disc*** modules)

This option introduces $R_i$ **throughout all liquid macro regions adjacent to the wall**.  
When both meso and macro resistances are activated simultaneously, the simulation remains numerically stable and **eliminates flux jumps** between meso and macro regions — although, as noted by *Urbano*, this configuration may not be physically consistent.

**Parameter:**
- `Ri_liq 3.7e-6` — thermal resistance value  

---



# Test case


## Objective
This 2D-axi simulation is designed to demenstrate the effect of interfacial thermal resistance, in micro-layer simulations.


## Summary of Initial and Boundary Conditions for fluid:
- **Boundary Conditions**
  - **Dynamics**: 
    - Outlet condition on the top and right sides.
    - Symmetry on the left side.
    - Wall type BC on the bottom
  - **Thermal**: 
    - Adiabatic (Symmetry) conditions on the left and right sides.
    - Conjugate heat transfer with solid substrate at the bottom
    - Fixed temperature $T_{{sat}} $ at the top right side.
  - **Phase**:
    - Apparent angle feeded from microregion model (see model.txt for details).
    - All other boundaries set to symmetry.
- **Initial Conditions:**
  - **Dynamics**: Initial velocity is zero throughout the domain.
  - **Thermal**: Restart from preparatory simulation `fluide.med`
  - **Phase**: Initial shape of vapor using $R_0$ and $\theta_0$: $ r^2+(z-R_0cos(\theta_0))^2=R_0^2 $.


## Summary of Initial and Boundary Conditions (thermal) for solid:
- **Boundary Conditions**
  - Adiabatic conditions on the left, bottom and right sides.
  - Conjugate heat transfer with fluid domain at the top
- **Initial Conditions**
  -  Restart from preparatory simulation `solide.med`
 
- **Volume Source at the Solid–Fluid Interface**
   
    A **volume source** is applied at the top layer of solid cells, simulating the very thin surface heat flux at the solid–fluid interface.
  
    This approach neglects the **heater thickness** and the **effect of its thermal properties**, effectively imposing the heat flux directly at the interface.

![ICs and BCs for Stefan Problem](src/figs/interfacial_resistance.png)

## Case Specifications
- ** Reduced Domain Length**: $ W  = 0.1\, \text{mm} $, $ H  = 0.1\, \text{mm} $ and $ L  = 1\, \text{mm} $
- **Initial Interface Position**: $R_0 = 20 \, \mu m$ and $ \theta_0 = 90 \, ^\circ$ at $ t = 0 \, \text{s} $
- **Material Properties**: Properties for water under atmospheric conditions:

|               | $\rho$ $kg/m^3$| $\mu$ $Pa\cdot s$        | $\lambda$ $W/(m\cdot K)$ | $C_p$ $J/(kg\cdot K)$   |
|---------------|----------------|--------------------------|--------------------------|-------------------------|
| **Liquid**    | 958.37         | $2.8 \times 10^{-4}$     | 0.679                    | $4.21 \times 10^3$      |
| **Vapor**     | 0.597          | $1.227 \times 10^{-5}$   | 0.025                    | $2.077 \times 10^3$     |
| **solid**     | 3980           |       -                  | 25.1                     | 929                     |
| $\sigma = 0.0589$ N/m, $\mathcal{L} = 2.256 \times 10^6$ J/kg                                                  | 

---

In [ ]:
from trustutils import run
import numpy as np

# Declaration of Author and (optionally) date 
run.introduction("L. WEI and TrioCFD team","31/10/2025")

# Declaration of the TRUST version
run.TRUST_parameters("v1.9.7_beta")

In [ ]:
import math
import os

from math import sqrt, pi, floor, log10
def dt_popinet(dx):
    Cp_v = 2.077e3   # Specific heat capacity of vapor (J/kg·K)
    lambda_v = 0.025  # Thermal conductivity of vapor (W/m·K)
    h_lg = 2.256e6
    
    rho_l = 958.37
    Cp_l=4.21e3
    rho_v = 0.597
    sigma = 5.89e-2
    lambda_l = 0.679  

    rho_m = (rho_v+rho_l)/2.
    n_sig_figs = 2
    dt = sqrt((rho_m/pi/sigma)*(dx)**3)
    scale = -int(floor(log10(abs(dt))) - (n_sig_figs - 1))
    return round(dt, scale)


# read one single TEMP file==========================================================================
import pandas as pd
def T_data_frame(file_path):
    data_list = []
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith(('Time', '-', '\n')):
                continue
            parts = [part.strip('|').strip() for part in line.split('\t')]
            try:
                time, x, y, twall = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                data_list.append((time, x, y, twall))
            except ValueError:
                continue

    df_parsed = pd.DataFrame(data_list, columns=['Time', 'X', 'Y', 'Twall'])
    return df_parsed

# heat flux ===========================================================================================
# Function to extract time from header lines
def extract_time(line):
    start = line.find("au temps") + 9
    end = line.find(":", start)
    time_str = line[start:end].strip()
    try:
        return float(time_str)
    except ValueError:
        return None
    
# Function to robustly parse a line of data
def parse_line_robust(line):
    # Extracting numerical values following specific keywords
    x_index = line.find('x=') + 2
    surface_face_index = line.find('surface_face(m2)=') + 17
    flux_par_surface_index = line.find('flux_par_surface(W/m2)=') + 23

    x = float(line[x_index:line.find('y=', x_index)].strip())
    surface_face = float(line[surface_face_index:line.find('flux_par_surface', surface_face_index)].strip())
    flux_par_surface = float(line[flux_par_surface_index:line.find('flux(W)=', flux_par_surface_index)].strip())

    return x, surface_face, flux_par_surface


def P_data_frame(input_filepath):
    num_line_effective = 0
    Times = []
    faces_filepath = "face.txt"
    # Open the input file for reading
    with open(input_filepath, 'r') as file:
        # Open the faces file for writing lines that start with '# Face'
        with open(faces_filepath, 'w') as faces_file:
            # Open the rest file for writing all other lines
            # Iterate over each line in the input file
            for line in file:
                # Write lines starting with '# Face' to the faces file
                if line.startswith('# Face'):
                    faces_file.write(line)
                    num_line_effective = num_line_effective + 1
                # Write all other lines to the rest file
                else:
                    time = extract_time(line)
                    Times.append(time)                
    unique_elements = list(set(Times))
    average = int(num_line_effective/np.array(unique_elements).size)
    unique_times = np.sort(np.array(unique_elements, dtype=float))
     
    
    data = [] 
    with open(faces_filepath, 'r') as file:
        for line in file:
            parsed_data = parse_line_robust(line)
            if parsed_data:
                data.append(parsed_data)

    os.remove(faces_filepath)
    df = pd.DataFrame(data, columns=['x', 'surface_face', 'flux_par_surface'])
    # Assign time based on the index of each row and the average number of lines per time
    df['Time'] = unique_times[df.index // int(average)]
    return df

In [ ]:
import os

run.reset()

dx = 1e-6
rmax = 0.1e-3
zmax = 0.1e-3
zsol = 1.e-3

Nx = int(rmax/dx)+1
Ny = int(zmax/dx)+1
Ns = int(zsol/dx)+1

Rinjection = 20.e-6

nprocx = 2
nprocy = 3

theta = 12.01
dt = dt_popinet(dx)*0.2
pws = 425.e3


R_M1 = np.asarray([0, 3.7e-4, 3.7e-4])
R_M2 = np.asarray([0, 0, 3.7e-4])
Case_name = ['Ref', 'Meso', 'MesoMacro']


for i in range(len(Case_name)):
    fname = f'{Case_name[i]}'
    name = 'source'
    substitutions_dict = {"rmax" : f'{rmax}',
                          "zmax" : f'{zmax}',
                          "zsol" : f'{zsol}',
                          "Nx" : str(Nx),
                          "Ny" : str(Ny),
                          "Ns" : str(Ns),
                          "nprocx" : str(nprocx),
                          "nprocy" : str(nprocy),
                          "timestep" : f'{dt}',
                          "Rinjection" : f'{Rinjection:.4g}',
                          "theta" : f'{theta:.3g}',
                          "pws" : f'{pws:.3g}',
                          "Ri_M1" : f'{R_M1[i]:.3g}',
                          "Ri_M2" : f'{R_M2[i]:.3g}',
                          "dx" : f'{dx:.3g}'
                          }

    tc = run.addCaseFromTemplate("source.data"
                             ,targetDirectory=f"{fname}"
                             ,dic=substitutions_dict
                             ,nbProcs=nprocx*nprocy
                             ,targetData=f"{name}.data"
                             )

    if (nprocx*nprocy > 1):
        # print("PARALLEL")
        tc.partition()
            
run.printCases()

In [ ]:
# Run all cases
run.runCases()

# Results
 


In [ ]:
def get_position_bis(fname, name, biaxi=True):
    import numpy as np
    import os, math
    out = f"{fname}/vap_vol.txt"
    if not (os.path.exists(f"{run.BUILD_DIRECTORY}/{out}") and os.path.getsize(f"{run.BUILD_DIRECTORY}/{out}") > 0) :
        os.system(f'grep "^Volume_phase_0" {run.BUILD_DIRECTORY}/{fname}/PAR_{name}.err | awk \'{{print $4, $2}}\' > {run.BUILD_DIRECTORY}/{out}')
    run.saveFileAccumulator(out)

    
    data = np.loadtxt(f'{run.BUILD_DIRECTORY}/{out}')
    
    # print(data)
    time = data[:,0]
    vol = data[:, 1] 
    if biaxi:
        posi = (np.array(vol)*3./2./math.pi)**(1./3.)
    else:
        posi = (np.array(vol)*4./math.pi)**(1./2.)
    return time, posi
def slice_with_last_array(data, freq):
    return np.concatenate((data[::freq], data[-1:] if data[-1] not in data[::freq] else []))

def get_prb_data(file, time):
    from trustutils.files import SonSEGFile
    
    son_file = file
    donne = SonSEGFile(son_file,None)
    compo = 0
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    
    # y_label = entries[compo].split()[0]
    # x_label = donne.getXLabel()
    
    # print("x_label : ", x_label)
    # print("y_label : ", y_label)
    
    # start, end = donne.getXTremePoints()
    # print("dom xmin and ymin : ", start)
    # print("dom xmax and ymax : ", end)
    
    t = donne.getValues(entries[0])[0]

    ## find closest value of t
    if time == None:
        idx = -1
    else:
        idx = (np.abs(t - time)).argmin()
        
    X = donne.getXAxis()
    
    Y = []
    for i in entries[compo::ncompo]:
        Y.append(list(donne.getValues(i)[1])[idx])
    if X[0] != X.min():
        X = X[::-1]
        Y = Y[::-1]
    return X, Y

def process_data(data, mirror = False):
   
    # Find the index of the maximum value in the second column
    max_x_index = np.argmax(data[:, 0])
    
    # Get the corresponding value from the third column
    corresponding_y = data[:, 1][max_x_index]
    
    # First part: Filter and sort data where the third column is less than corresponding_y
    mask1 = data[:, 1] < corresponding_y
    x_filtered1 = data[:, 0][mask1]
    y_filtered1 = data[:, 1][mask1]
    xy_pairs1 = list(zip(x_filtered1, y_filtered1))
    xy_pairs_sorted1 = sorted(xy_pairs1, key=lambda pair: pair[0])
    x1, y1 = zip(*xy_pairs_sorted1)
    
    # Second part: Filter and sort data where the third column is greater than or equal to corresponding_y
    mask2 = data[:, 1] >= corresponding_y
    x_filtered2 = data[:, 0][mask2]
    y_filtered2 = data[:, 1][mask2]
    xy_pairs2 = list(zip(x_filtered2, y_filtered2))
    xy_pairs_sorted2 = sorted(xy_pairs2, key=lambda pair: -pair[0])
    x2, y2 = zip(*xy_pairs_sorted2)
    
    # Concatenate the results
    x = np.asarray(x1 + x2)
    y = np.asarray(y1 + y2)

    if mirror:
        x3 = -np.asarray(x2)
        y3 = np.asarray(y2)
        x3 = x3[::-1]
        y3 = y3[::-1]

        x4 = -np.asarray(x1)
        y4 = np.asarray(y1)
        x4 = x4[::-1]
        y4 = y4[::-1]
        
        x=np.concatenate((x, x3, x4))
        y=np.concatenate((y, y3, y4))
        
    
    return x, y
 

def get_time(file_path):
    time = [ ]
    with open(file_path+'/post.lata', 'r') as file:
            lines = file.readlines()
    for line in lines:
        if re.search(r'TEMPS', line):
            # Split the line by spaces and get the second element
            parts = line.split()
            if len(parts) > 1:
                second_element = parts[1]
                time.append(second_element)
                # print(second_element)
    return np.asarray(time).astype(float)

colors = ['red', 'blue', 'green', 'black', 'orange', 'purple']
Mar_scatter = ['o', 's', '*', 'D','H']
size_sym = 30

## Radius vs. time

In [ ]:
import subprocess
import os

for index in range(len(Case_name)):
    case_dir = os.path.join(run.BUILD_DIRECTORY, Case_name[index])
    exe_path = os.path.join("..", run.BUILD_DIRECTORY, "post_run")

    # Run `../build/post_run PAR_source` inside the case directory
    subprocess.run([exe_path, "PAR_source"], cwd=case_dir, check=True)

In [ ]:
import matplotlib.pyplot as plt

for i in range(len(Case_name)):
    data = np.loadtxt(f'{run.BUILD_DIRECTORY}/{Case_name[i]}/dia_full.txt')
    plt.scatter(data[::50, 0]*1.e3, data[::50, 1]*1.e6
                , marker=Mar_scatter[i%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors=colors[i%len(Case_name)]

                # , color =colors[i%len(Case_name)]
                ,   label=f'{Case_name[i]}')
    data = np.loadtxt(f'{run.BUILD_DIRECTORY}/{Case_name[i]}/base_dia.txt', skiprows=1)
    plt.scatter(data[::10, 0]*1.e3, data[::10, 1]*1.e6
                , marker=Mar_scatter[i%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors=colors[i%len(Case_name)]
                # , color =colors[i%len(Case_name)]
               )
        
        
# plt.xlim([1,1.4])
# plt.ylim([1,1.2])

plt.xlabel(r"$t \ ( \mu s)$")
plt.ylabel(r"$R_{eq},  \  r_{cl} \ ( \mu m)$")
plt.legend(loc="best")
# plt.title(itemC)
plt.show()

## Interface shape at end of simu

In [ ]:
t_cible = 5e-6

In [ ]:
import re, glob
fig, ax = plt.subplots()

for index in range(len(Case_name)):
    path = f'{run.BUILD_DIRECTORY}/{Case_name[index]}/lata'
    
    time = get_time(f'{path}')
    tsp_ind = np.abs(time - t_cible).argmin() 
    my_file = f'{path}/post-interf-ascii_post.lata.INTERFACES.{time[tsp_ind]:.6f}*'

    matching_files = glob.glob(my_file)
    my_file = matching_files[1]
    if my_file.endswith('.elem'):
         my_file = matching_files[0]
    
    with open(my_file, 'r') as file:
        data_array = file.readlines()[1:-1]
    data = np.loadtxt(data_array)
    x, y = process_data(data, True)

    ax.plot( x*1.e3, y*1.e3, color =colors[index%len(Case_name)], label=f'{Case_name[index]}')



# Adding titles and labels
# ax.set_title("Mirrored Plot across the Y-Axis")
# ax.set_xlabel("r (mm)", fontweight='bold')
# ax.set_ylabel("z (mm)", fontweight='bold')
ax.set_xlabel(r"$r$ (mm)")
ax.set_ylabel(r"$z$ (mm)")
ax.set_aspect('equal')
ax.set_xlim([0, 0.04])
ax.set_ylim([0, 0.04])
# ax.set_ylim([0, 2])
plt.legend(loc='best')
# plt.legend(loc="center left", bbox_to_anchor=(-1.25, 0.5))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
# plt.axis('equal')
# plt.savefig(f'{save_path}/contour_tcl4.png', bbox_inches='tight')
# Show the plot
plt.show()

## ML

In [ ]:
fig, ax = plt.subplots()

for index in range(len(Case_name)):
    path = f'{run.BUILD_DIRECTORY}/{Case_name[index]}/lata'
    time = get_time(f'{path}')
    tsp_ind = np.abs(time - t_cible).argmin() 
    my_file = f'{path}/post-interf-ascii_post.lata.INTERFACES.{time[tsp_ind]:.6f}*'

    matching_files = glob.glob(my_file)
    my_file = matching_files[1]
    if my_file.endswith('.elem'):
         my_file = matching_files[0]
    
    with open(my_file, 'r') as file:
        data_array = file.readlines()[1:-1]
    data = np.loadtxt(data_array)
    x, y = process_data(data, True)

    ax.scatter( x*1.e6, y*1.e6
               , marker=Mar_scatter[index%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors='k'
               # , color =colors[index%len(Case_name)]
               ,  label=f'{Case_name[index]}')


# Adding titles and labels
# ax.set_title("Mirrored Plot across the Y-Axis")
# ax.set_xlabel("r (mm)", fontweight='bold')
# ax.set_ylabel("z (mm)", fontweight='bold')
plt.xlabel(r'$\mathrm{r} \ (\mu m)$')
ax.set_ylabel(r"$z$ (mm)")
# ax.set_aspect('equal')
ax.set_xlim([20, 35])
ax.set_ylim([0, 9])
# ax.set_ylim([0, 2])
plt.legend(loc='best')
# plt.legend(loc="center left", bbox_to_anchor=(-1.25, 0.5))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
# plt.axis('equal')
# plt.savefig(f'{save_path}/contour_tcl4.png', bbox_inches='tight')
# Show the plot
plt.show()

## Wall heat flux

In [ ]:
fig, ax = plt.subplots()

for index in range(len(Case_name)-1):
    fname = f'{Case_name[index]}'
    file_name = 'PAR_source_pb2_Diffusion_chaleur.face'
    file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
    df_pwall = P_data_frame(file_path)
    # print(len(df_pwall)%(Nx-1))
    time_pwall = df_pwall['Time'].unique()

    time_pwall = df_pwall['Time'].unique()
    df_parsed = df_pwall
    unique_time_steps = time_pwall
    tsp_ind = np.abs(unique_time_steps - t_cible).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.plot(data_at_time_step['x']*1.e6, -data_at_time_step['flux_par_surface']/1.e6 
             #   , marker=Mar_scatter[index%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors='k'
             ,color =colors[index%len(Case_name)]
                ,   label= f'{fname}'
            )
    # print(time_step)

for index in [2]:
    fname = f'{Case_name[index]}'
    file_name = 'PAR_source_pb2_Diffusion_chaleur.face'
    file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
    df_pwall = P_data_frame(file_path)
    # print(len(df_pwall)%(Nx-1))
    time_pwall = df_pwall['Time'].unique()

    time_pwall = df_pwall['Time'].unique()
    df_parsed = df_pwall
    unique_time_steps = time_pwall
    tsp_ind = np.abs(unique_time_steps - t_cible).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.scatter(data_at_time_step['x']*1.e6, -data_at_time_step['flux_par_surface']/1.e6 
                , marker=Mar_scatter[index%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors='k'
             # ,color =colors[index%len(Case_name)]
                ,   label= f'{fname}'
            )

plt.xlabel(r'$\mathrm{r} \ (\mu m)$')
plt.ylabel(r'$\mathrm{\phi _{wall}\ (MW/m^2)}$')
# plt.ylim(1e-9, 9.e-3)
# plt.xlim(0, 0.25)
# plt.yscale('log')
plt.title(f'Heat flux in Meso region at {time_step*1.e6:.3g} ' +r'$\mu s$')
plt.xlim(20, 35)
# plt.ylim(-0.01, 0.01)
plt.legend( )
# plt.yscale('log')
# plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
plt.show()


In [ ]:

fig, ax = plt.subplots()

for index in range(len(Case_name)):
    fname = f'{Case_name[index]}'
    file_name = 'PAR_source_pb2_Diffusion_chaleur.face'
    file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
    df_pwall = P_data_frame(file_path)
    # print(len(df_pwall)%(Nx-1))
    time_pwall = df_pwall['Time'].unique()

    time_pwall = df_pwall['Time'].unique()
    df_parsed = df_pwall
    unique_time_steps = time_pwall
    tsp_ind = np.abs(unique_time_steps - t_cible).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.scatter(data_at_time_step['x']*1.e6, -data_at_time_step['flux_par_surface']/1.e3, 
                marker=Mar_scatter[index%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors='k'
             #         , color =colors[index%len(Case_name)]
                      ,  label= f'{fname}'
            )


plt.xlabel(r'$\mathrm{r\ (mm)}$')
plt.ylabel(r'$\mathrm{\phi _{wall}\ (kW/m^2)}$')
# plt.ylim(0., 50.)
plt.xlim(60, 80)
# plt.yscale('log')
plt.title(f'Heat flux in Macro region at {time_step*1.e6:.3g} ' +r'$\mu s$')
plt.ylim(0, 2.5)
plt.legend( )
# plt.yscale('log')
# plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
plt.show()

## Wall temperature

In [ ]:
fig, ax = plt.subplots()

for index in range(len(Case_name)-1):
    fname = f'{Case_name[index]}'
    file_name = 'PAR_source_pb2_sup_twall.face'
    
    file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
    df_twall = T_data_frame(file_path)
    # print(len(df_twall)%(Nx-1))
    time_twall = df_twall['Time'].unique()

    unique_time_steps = time_twall
    df_parsed = df_twall
    
    tsp_ind = np.abs(unique_time_steps - t_cible).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.plot(data_at_time_step['X']*1.e3, data_at_time_step['Twall']       
             , color =colors[index%len(Case_name)],  label= f'{fname}'
            )

for index in [2]:
    fname = f'{Case_name[index]}'
    file_name = 'PAR_source_pb2_sup_twall.face'
    
    file_path = f'{run.BUILD_DIRECTORY}/{fname}/{file_name}'
    df_twall = T_data_frame(file_path)
    # print(len(df_twall)%(Nx-1))
    time_twall = df_twall['Time'].unique()

    unique_time_steps = time_twall
    df_parsed = df_twall
    
    tsp_ind = np.abs(unique_time_steps - t_cible).argmin()
    time_step = unique_time_steps[tsp_ind]
    data_at_time_step = df_parsed[df_parsed['Time'] == time_step]
    plt.scatter(data_at_time_step['X']*1.e3, data_at_time_step['Twall']       
             # , color =colors[index%len(Case_name)]
            , marker=Mar_scatter[index%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors='k'
             ,  label= f'{fname}'
            )


plt.xlabel(r'$r$ (mm)')
plt.ylabel(r'$\Delta T^{w}$  (K)' )



plt.xlim(0, 0.1)
plt.ylim(11.75, 12.5)

plt.title(f'{time_step*1.e6:.3g} ' +r'$\mu s$')
# plt.ylim(0, 7)
plt.legend( )
# plt.yscale('log')
# plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
# plt.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.55))
plt.show()


## Temeparature in the vertical

In [ ]:
from scipy.interpolate import interp1d

time_son = t_cible

fig, ax = plt.subplots()

for i in range(len(Case_name)):
    fname = f'{Case_name[i]}'
    son_file = f'{run.BUILD_DIRECTORY}/{fname}/PAR_source_T_VERT.son'
    XL, YL = get_prb_data(son_file, time_son)
    
    son_file = f'{run.BUILD_DIRECTORY}/{fname}/PAR_source_T_VERT_SOL.son'
    XS, YS = get_prb_data(son_file, time_son)

    f = interp1d(XS, YS, kind='linear', fill_value='extrapolate')  
    x_new = 0
    y_new = f(x_new)
    XS_new = np.append(XS, x_new)
    YS_new = np.append(YS, y_new)
    
    Xsimu = np.concatenate((XS_new, XL))
    Ysimu = np.concatenate((YS_new, YL))
    
    ax.scatter(Xsimu*1.e6, Ysimu
            , marker=Mar_scatter[i%len(Mar_scatter)], s=size_sym, facecolors='none', edgecolors='k'
            # , color =colors[i%len(Case_name)]
            ,  label=f'{Case_name[i]}')
  
plt.axvline(x=0, color='r', linestyle='--', linewidth=2, label='wall')

ax.set_xlim([-5, 4])
ax.set_ylim([12.2, 12.45])

plt.xlabel(r'$z$ (mm)')
plt.ylabel(r'$\Delta T$  (K)' )
# plt.yscale('log')

plt.title(f'r=66 '+ r'$\mu m$ - '+f'{(time_son)*1.e6:.3g} ' +r'$\mu s$')
ax.legend()
plt.show()

# Data set

In [ ]:
run.dumpDatasetMD(f"{run.BUILD_DIRECTORY}/{Case_name[0]}/PAR_{name}.data")